# 10 — Advanced RAG with SochDB

This notebook covers **advanced Retrieval-Augmented Generation** patterns using SochDB:

| Feature | Description |
|---|---|
| **Multi-Vector Documents** | Store multiple embeddings per document (chunks) and aggregate during search |
| **HybridRetriever** | Unified retrieval API with AllowedSet pre-filtering and explain() |
| **Metadata Filtering** | Filter results by metadata fields during search |
| **Exact Brute-Force Search** | Compare ANN vs exact recall@k |
| **SearchRequest** | Advanced search surface with all parameters |
| **Gemini-Powered RAG** | End-to-end question answering with Gemini + SochDB retrieval |

In [8]:
import os, json, time, shutil
from openai import OpenAI
from sochdb import Database
from sochdb.namespace import CollectionConfig, SearchRequest
from sochdb.memory.retrieval import (
    HybridRetriever, RetrievalConfig, AllowedSet,
    FFIRetrievalBackend,
)

# Gemini via OpenAI-compatible SDK
client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

EMBED_MODEL = "gemini-embedding-001"  # 3072-dim
CHAT_MODEL  = "gemini-2.5-flash-lite"
DIM = 3072

def embed(text: str) -> list[float]:
    """Get Gemini embedding for a single text."""
    resp = client.embeddings.create(model=EMBED_MODEL, input=text)
    return resp.data[0].embedding

def embed_batch(texts: list[str]) -> list[list[float]]:
    """Get Gemini embeddings for a batch of texts."""
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in resp.data]

print(f"Embedding dimension: {len(embed('test'))}")

Embedding dimension: 3072


## 1. Setup Database & Collection

In [9]:
DB_PATH = "./advanced_rag_db"
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)

db = Database.open(DB_PATH)
ns = db.create_namespace("rag_demo")

# Create collection with 3072-dim for Gemini embeddings
config = CollectionConfig(name="docs", dimension=DIM, m=16, ef_construction=200)
collection = ns.create_collection(config)
collection.set_ef_search(500)  # High ef_search for better recall

print(f"Collection created: {collection.name} (dim={DIM})")

Collection created: docs (dim=3072)


## 2. Ingest Documents with Rich Metadata

We'll ingest a small knowledge base with metadata (category, source, year) so we can demonstrate metadata-filtered search later.

In [10]:
documents = [
    {
        "id": "doc_ml_1",
        "text": "Transformers use self-attention mechanisms to process sequences in parallel, replacing recurrent architectures. They were introduced in the 2017 paper 'Attention Is All You Need' by Vaswani et al.",
        "metadata": {"category": "machine_learning", "source": "arxiv", "year": 2017}
    },
    {
        "id": "doc_ml_2",
        "text": "BERT (Bidirectional Encoder Representations from Transformers) is a pre-trained language model that uses masked language modeling. It excels at understanding context by reading text bidirectionally.",
        "metadata": {"category": "machine_learning", "source": "google_ai", "year": 2018}
    },
    {
        "id": "doc_ml_3",
        "text": "GPT-4 is a large multimodal model that accepts both image and text inputs. It demonstrates human-level performance on various professional and academic benchmarks.",
        "metadata": {"category": "machine_learning", "source": "openai", "year": 2023}
    },
    {
        "id": "doc_db_1",
        "text": "SochDB is an embedded vector database written in Rust. It supports hybrid search (vector + BM25 keyword), ACID transactions, temporal graphs, and priority queues.",
        "metadata": {"category": "databases", "source": "sochdb", "year": 2025}
    },
    {
        "id": "doc_db_2",
        "text": "ChromaDB is an open-source embedding database for building AI applications. It features simple API, multi-modal support, and integrations with LangChain and LlamaIndex.",
        "metadata": {"category": "databases", "source": "chroma", "year": 2023}
    },
    {
        "id": "doc_db_3",
        "text": "Pinecone is a managed vector database service for building real-time recommendation and search systems. It offers automatic scaling and hybrid search capabilities.",
        "metadata": {"category": "databases", "source": "pinecone", "year": 2023}
    },
    {
        "id": "doc_rag_1",
        "text": "Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model. The retriever fetches relevant documents, and the generator produces answers grounded in those documents.",
        "metadata": {"category": "rag", "source": "meta_ai", "year": 2020}
    },
    {
        "id": "doc_rag_2",
        "text": "Advanced RAG techniques include hybrid search (combining dense and sparse retrieval), reranking with cross-encoders, query expansion, and hypothetical document embeddings (HyDE).",
        "metadata": {"category": "rag", "source": "research", "year": 2024}
    },
]

# Embed all documents
texts = [doc["text"] for doc in documents]
embeddings = embed_batch(texts)

# Insert using insert() which writes to both HNSW and KV store
for doc, emb in zip(documents, embeddings):
    collection.insert(
        id=doc["id"],
        vector=emb,
        metadata=doc["metadata"],
        content=doc["text"],
    )

print(f"Inserted {len(documents)} documents")

Inserted 8 documents


## 3. Basic Vector Search vs Exact Search

Compare HNSW approximate nearest neighbour search with exact brute-force search.

In [11]:
import numpy as np

query = "What is a transformer model?"
query_vec = embed(query)

# --- ANN (HNSW) search ---
ann_results = collection.vector_search(vector=query_vec, k=5)
print("=== ANN (HNSW) Search ===")
for r in ann_results:
    meta = r.metadata or {}
    print(f"  {r.id:12s}  score={r.score:.4f}  [{meta.get('category','')}]")

# --- Exact brute-force search (pure Python fallback) ---
# Native FFI exact search can segfault on high-dim vectors,
# so we use a reliable Python cosine-similarity implementation.
def exact_search(collection, query_vec, k=5):
    """Pure-Python brute-force kNN using cosine similarity."""
    q = np.array(query_vec, dtype=np.float32)
    q_norm = np.linalg.norm(q)
    scores = []
    for doc_id, raw_vec in collection._raw_vectors.items():
        v = np.array(raw_vec, dtype=np.float32)
        v_norm = np.linalg.norm(v)
        if q_norm > 0 and v_norm > 0:
            sim = float(np.dot(q, v) / (q_norm * v_norm))
        else:
            sim = 0.0
        scores.append((doc_id, sim))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]

exact_results = exact_search(collection, query_vec, k=5)
print("\n=== Exact Brute-Force Search (Python) ===")
for doc_id, score in exact_results:
    print(f"  {doc_id:12s}  score={score:.4f}")

# --- Compare rankings ---
ann_ids = [r.id for r in ann_results]
exact_ids = [doc_id for doc_id, _ in exact_results]
print(f"\nRankings match: {ann_ids == exact_ids}")

=== ANN (HNSW) Search ===
  doc_ml_1      score=0.7388  [machine_learning]
  doc_ml_2      score=0.6400  [machine_learning]
  doc_rag_1     score=0.5978  [rag]
  doc_ml_3      score=0.5904  [machine_learning]
  doc_rag_2     score=0.5601  [rag]

=== Exact Brute-Force Search (Python) ===
  doc_ml_1      score=0.7388
  doc_ml_2      score=0.6400
  doc_rag_1     score=0.5978
  doc_ml_3      score=0.5904
  doc_rag_2     score=0.5601

Rankings match: True


## 4. Multi-Vector Documents (Chunked RAG)

For long documents, you can store multiple chunk embeddings per document using `insert_multi()`. During search, chunk-level scores are **aggregated** (`max`, `mean`, or `first`) into a single document-level score.

In [12]:
# Create a separate collection for multi-vector demo
mv_config = CollectionConfig(name="chunked_docs", dimension=DIM)
mv_collection = ns.create_collection(mv_config)

# Simulate a long document split into chunks
long_doc_chunks = [
    "Chapter 1: Vector databases store high-dimensional embeddings and enable fast similarity search using algorithms like HNSW.",
    "Chapter 2: SochDB combines vector search with BM25 keyword matching using Reciprocal Rank Fusion (RRF) for hybrid retrieval.",
    "Chapter 3: Temporal graphs in SochDB allow tracking how relationships change over time, supporting time-travel queries.",
]

# Embed each chunk
chunk_embeddings = embed_batch(long_doc_chunks)

# Insert as multi-vector document
mv_collection.insert_multi(
    id="long_doc_1",
    vectors=chunk_embeddings,
    metadata={"title": "SochDB Deep Dive", "author": "Sushanth"},
    chunk_texts=long_doc_chunks,
    aggregate="max",  # Use the best chunk score as document score
)

print(f"Stored multi-vector document with {len(long_doc_chunks)} chunks")
print(f"Aggregation method: max (best chunk score wins)")

Stored multi-vector document with 3 chunks
Aggregation method: max (best chunk score wins)


## 5. Metadata-Filtered Search

Search with metadata filters to scope results to specific categories, time ranges, or sources.

In [13]:
query = "What databases support vector search?"
query_vec = embed(query)

# Unfiltered search
print("=== Unfiltered Search ===")
results = collection.vector_search(vector=query_vec, k=5)
for r in results:
    meta = r.metadata or {}
    print(f"  {r.id:12s}  score={r.score:.4f}  category={meta.get('category','?')}")

# Filtered: only databases category
print("\n=== Filtered: category=databases ===")
results_filtered = collection.vector_search(
    vector=query_vec, k=5, filter={"category": "databases"}
)
for r in results_filtered:
    meta = r.metadata or {}
    print(f"  {r.id:12s}  score={r.score:.4f}  source={meta.get('source','?')}")

# Filtered: only 2023+ results
print("\n=== Filtered: year=2023 ===")
results_2023 = collection.vector_search(
    vector=query_vec, k=5, filter={"year": 2023}
)
for r in results_2023:
    meta = r.metadata or {}
    print(f"  {r.id:12s}  score={r.score:.4f}  year={meta.get('year','?')}")

=== Unfiltered Search ===
  doc_db_3      score=0.6787  category=databases
  doc_db_1      score=0.6740  category=databases
  doc_db_2      score=0.6474  category=databases
  doc_rag_1     score=0.6146  category=rag
  doc_rag_2     score=0.6108  category=rag

=== Filtered: category=databases ===
  doc_db_3      score=0.6787  source=pinecone
  doc_db_1      score=0.6740  source=sochdb
  doc_db_2      score=0.6474  source=chroma

=== Filtered: year=2023 ===
  doc_db_3      score=0.6787  year=2023
  doc_db_2      score=0.6474  year=2023
  doc_ml_3      score=0.5290  year=2023


## 6. HybridRetriever with AllowedSet Pre-Filtering

The `HybridRetriever` provides a unified API with **security-by-construction**: results are pre-filtered via an `AllowedSet` so that multi-tenant apps never leak documents across tenants.

In [14]:
# Create HybridRetriever from the database
retriever = HybridRetriever.from_database(
    db,
    namespace="rag_demo",
    collection="docs",
    config=RetrievalConfig(
        k=5,
        alpha=0.7,    # 70% vector, 30% keyword
        rrf_k=60,
    ),
)

query_text = "hybrid search vector database"
query_vector = embed(query_text)

# --- AllowedSet: only specific document IDs ---
allowed_ids = AllowedSet.from_ids(["doc_db_1", "doc_db_2", "doc_db_3", "doc_rag_2"])

response = retriever.retrieve(
    query_text=query_text,
    query_vector=query_vector,
    allowed=allowed_ids,
    k=3,
)

print(f"=== HybridRetriever with AllowedSet (IDs) ===")
print(f"Total candidates: {response.total_candidates}")
print(f"Filtered out:     {response.filtered_count}")
print(f"Query time:       {response.query_time_ms:.1f}ms")
print(f"Results:")
for r in response.results:
    print(f"  {r.id:12s}  score={r.score:.6f}")

=== HybridRetriever with AllowedSet (IDs) ===
Total candidates: 8
Filtered out:     4
Query time:       8.5ms
Results:
  doc_rag_2     score=0.016237
  doc_db_3      score=0.016129
  doc_db_1      score=0.016029


In [15]:
# --- AllowedSet: namespace prefix ---
allowed_ns = AllowedSet.from_namespace("doc_ml")

response_ml = retriever.retrieve(
    query_text="language model architecture",
    query_vector=embed("language model architecture"),
    allowed=allowed_ns,
    k=5,
)

print("=== AllowedSet: namespace prefix 'doc_ml' ===")
print(f"Filtered out: {response_ml.filtered_count} (non-ML docs removed)")
for r in response_ml.results:
    print(f"  {r.id:12s}  score={r.score:.6f}")

=== AllowedSet: namespace prefix 'doc_ml' ===
Filtered out: 5 (non-ML docs removed)
  doc_ml_2      score=0.016393
  doc_ml_1      score=0.015873
  doc_ml_3      score=0.015625


In [16]:
# --- AllowedSet: custom filter function ---
# Only allow documents from 2023 or later
allowed_recent = AllowedSet(
    filter_fn=lambda doc_id, meta: meta.get("year", 0) >= 2023
)

response_recent = retriever.retrieve(
    query_text="latest AI models",
    query_vector=embed("latest AI models"),
    allowed=allowed_recent,
    k=5,
)

print("=== AllowedSet: custom filter (year >= 2023) ===")
print(f"Filtered out: {response_recent.filtered_count}")
for r in response_recent.results:
    print(f"  {r.id:12s}  score={r.score:.6f}  metadata={r.metadata}")

=== AllowedSet: custom filter (year >= 2023) ===
Filtered out: 3
  doc_ml_3      score=0.016237  metadata={'category': 'machine_learning', 'source': 'openai', 'year': 2023, '_content': 'GPT-4 is a large multimodal model that accepts both image and text inputs. It demonstrates human-level performance on various professional and academic benchmarks.'}
  doc_db_2      score=0.015856  metadata={'category': 'databases', 'source': 'chroma', 'year': 2023, '_content': 'ChromaDB is an open-source embedding database for building AI applications. It features simple API, multi-modal support, and integrations with LangChain and LlamaIndex.'}
  doc_rag_2     score=0.010769  metadata={'category': 'rag', 'source': 'research', 'year': 2024, '_content': 'Advanced RAG techniques include hybrid search (combining dense and sparse retrieval), reranking with cross-encoders, query expansion, and hypothetical document embeddings (HyDE).'}
  doc_db_3      score=0.010448  metadata={'category': 'databases', 'sour

## 7. Explain: Debug Why a Document Ranked Where It Did

The `explain()` method shows vector rank, keyword rank, and expected RRF score for a specific document — essential for debugging retrieval quality.

In [17]:
query_text = "vector similarity search database"
query_vector = embed(query_text)

# Explain ranking for a specific document
explanation = retriever.explain(
    query_text=query_text,
    query_vector=query_vector,
    doc_id="doc_db_1",
)

print("=== Explain: doc_db_1 ===")
print(json.dumps(explanation, indent=2, default=str))

# Compare with another document
explanation2 = retriever.explain(
    query_text=query_text,
    query_vector=query_vector,
    doc_id="doc_ml_1",
)

print("\n=== Explain: doc_ml_1 ===")
print(json.dumps(explanation2, indent=2, default=str))

TypeError: search() got an unexpected keyword argument 'vector'

## 8. End-to-End RAG with Gemini

Combine SochDB retrieval with Gemini generation for a full RAG pipeline.

In [ ]:
def rag_answer(question: str, k: int = 3) -> str:
    """Full RAG pipeline: embed → retrieve → generate."""
    # 1. Embed the question
    q_vec = embed(question)
    
    # 2. Retrieve relevant documents using HybridRetriever
    response = retriever.retrieve(
        query_text=question,
        query_vector=q_vec,
        allowed=AllowedSet.allow_all(),
        k=k,
    )
    
    # 3. Build context from retrieved docs
    context_parts = []
    for i, result in enumerate(response.results, 1):
        meta = result.metadata or {}
        content = meta.get("_content", result.content or "")
        context_parts.append(f"[{i}] (score={result.score:.4f}) {content}")
    context = "\n\n".join(context_parts)
    
    # 4. Generate answer using Gemini
    system_prompt = (
        "You are a helpful assistant. Answer questions based ONLY on the provided context. "
        "If the context doesn't contain enough information, say so. "
        "Cite document numbers [1], [2], etc. in your answer."
    )
    
    user_prompt = f"Context:\n{context}\n\nQuestion: {question}"
    
    chat_response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
    )
    
    return chat_response.choices[0].message.content

# Test the RAG pipeline
questions = [
    "What is the difference between SochDB and ChromaDB?",
    "How does hybrid search work in vector databases?",
    "What is the transformer architecture?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    answer = rag_answer(q)
    print(f"A: {answer}")

## 9. SearchRequest: Full Control

For maximum control, use `SearchRequest` directly on the collection. It supports all parameters: `vector`, `text_query`, `filter`, `alpha`, `k`, `min_score`, `aggregate`, `as_of`, `include_vectors`, `include_metadata`, `include_scores`.

In [ ]:
# Build a SearchRequest with full control
request = SearchRequest(
    vector=embed("vector search database"),
    text_query="vector search",      # Enable hybrid (vector + BM25)
    k=4,
    alpha=0.6,                        # 60% vector, 40% keyword
    filter={"category": "databases"},  # Only database docs
    include_metadata=True,
    include_scores=True,
)

search_results = collection.search(request)

print(f"SearchRequest results ({search_results.query_time_ms:.1f}ms):")
for r in search_results:
    meta = r.metadata or {}
    print(f"  {r.id:12s}  score={r.score:.4f}  source={meta.get('source','?')}")

## 10. Cleanup

In [ ]:
db.close()
shutil.rmtree(DB_PATH, ignore_errors=True)
print("Done! Database cleaned up.")